<style>
  /* 1. Fix standard input code cells */
  pre, code, .jp-CodeCell .jp-Cell-inputWrapper {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
  }

  /* 2. Fix display() outputs and rendered Markdown */
  .jp-OutputArea-output, 
  .jp-OutputArea-output pre, 
  .jp-RenderedHTMLCommon pre, 
  .jp-RenderedMarkdown pre {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
      
      /* Forces the browser to print the overflow instead of hiding it in a scrollbox */
      overflow-x: visible !important; 
      max-width: 100% !important;
  }
</style>

In [1]:
%load_ext autoreload
%autoreload 2

### **Prediksi *Pavement Condition Index* menggunakan Model *Machine Learning (RF, XGBoost, ANN)***

#### 1. Pembersihan data dan perhitungan fitur density

In [2]:
# Memilih dataset dari menu
import pandas as pd
import ipywidgets as wg
from IPython.display import display, Markdown
from pci_ml.data import (list_datasets, load_data, clean_data, split_data,
                         checkpoint_options, load_checkpoint)

EXPORT_MODE = True
RELEASE_MEMORY = True

FILE_NAME = None        # None = first dataset, or e.g. "data_2026.xlsx"
N_SEED = 200
TEST_SIZE = 0.2
SHARE_TOL = 0.01
PCI_MIN = 0
CKPT_TAG = None         # None = newest split, or a checkpoint folder name

from pci_ml.memory import configure, release
configure(release=RELEASE_MEMORY)

datasets = list_datasets()
if FILE_NAME is not None and FILE_NAME not in datasets:
    raise ValueError(f"FILE_NAME '{FILE_NAME}' not found in datasets")

dropdown = wg.Dropdown(
    options=datasets,
    value=FILE_NAME or (datasets[0] if datasets else None),
    description="Pilih file :",
    layout=wg.Layout(width="400px"),
)
df = None

if EXPORT_MODE:
    print(f"File terpilih untuk analisis: {dropdown.value}")
else:
    display(dropdown)

# Helper function that adapts based on EXPORT_MODE
def format_adaptive_df(df_input, is_export_mode):
    if df_input is None:
        return ""      
    df_head = df_input.head()
    if not is_export_mode:
        return df_head.to_markdown(index=False)
    left_count = 5
    right_count = 2
    if df_head.shape[1] > (left_count + right_count):
        left_cols = df_head.iloc[:, :left_count]
        right_cols = df_head.iloc[:, -right_count:]
        middle_col = pd.DataFrame({"…": ["…"] * len(df_head)}, index=df_head.index)
        df_display = pd.concat([left_cols, middle_col, right_cols], axis=1)
    else:
        df_display = df_head
    markdown_table = df_display.to_markdown(index=False)
    return f'<div style="font-size: 11px;">\n\n{markdown_table}\n\n</div>'

handle = display(Markdown(""), display_id=True)

def on_select(change):
    name = change["new"]
    raw = None if name is None else load_data(name)

    markdown_table = format_adaptive_df(raw, EXPORT_MODE)
    handle.update(Markdown(markdown_table))

dropdown.observe(on_select, names="value")
on_select({"new": dropdown.value})

File terpilih untuk analisis: pci-ml-26.xlsx


<div style="font-size: 11px;">

| ROAD_NAME    | ROAD_ID   | SEGMENT_ID   |   SEGMENT_AREA |   GATOR_CRACK_L | …   |   RAVELLING_H |     PCI |
|:-------------|:----------|:-------------|---------------:|----------------:|:----|--------------:|--------:|
| Guru Bangkol | R001      | S001         |            450 |            0    | …   |             0 | 85.4969 |
| Guru Bangkol | R001      | S002         |            450 |            0    | …   |             0 | 73.3368 |
| Guru Bangkol | R001      | S003         |            450 |            0    | …   |             0 | 89.9305 |
| Guru Bangkol | R001      | S004         |            450 |            0.64 | …   |             0 | 73.6287 |
| Guru Bangkol | R001      | S005         |            450 |            0    | …   |             0 | 96.1874 |

</div>

In [3]:
# Membersihkan data dari duplikat, pencilan dan data tidak valid, serta menghitung fitur density
filename = dropdown.value
df, features = clean_data(load_data(filename))

markdown_table = format_adaptive_df(df, EXPORT_MODE)
display(Markdown(markdown_table))

<div style="font-size: 11px;">

| ROAD_NAME    | ROAD_ID   | SEGMENT_ID   |   GATOR_CRACK_L |   GATOR_CRACK_M | …   |   RAVELLING_H |     PCI |
|:-------------|:----------|:-------------|----------------:|----------------:|:----|--------------:|--------:|
| Guru Bangkol | R001      | S001         |      0          |      0          | …   |             0 | 85.4969 |
| Guru Bangkol | R001      | S002         |      0          |      0.00391778 | …   |             0 | 73.3368 |
| Guru Bangkol | R001      | S003         |      0          |      0          | …   |             0 | 89.9305 |
| Guru Bangkol | R001      | S004         |      0.00142222 |      0.0231111  | …   |             0 | 73.6287 |
| Guru Bangkol | R001      | S005         |      0          |      0          | …   |             0 | 96.1874 |

</div>

#### 2. Pemisahan data latih dan data uji

In [4]:
# Menentukan parameter dan kriteria pemisahan data
style = {"description_width": "100px"}
box = wg.Layout(width="180px", margin="0 15px")

n_seed = wg.IntText(value=N_SEED, description="Jumlah seed:", style=style, layout=box)
test_size = wg.FloatText(value=TEST_SIZE, description="Test size:", style=style, layout=box)
share_tol = wg.FloatText(value=SHARE_TOL, description="Tolerance:", style=style, layout=box)
pci_min = wg.FloatText(value=PCI_MIN, description="PCI minimum:", style=style, layout=box)

if EXPORT_MODE:
    print("Parameter Pemisahan Data:")
    print(f"- Jumlah seed     : {n_seed.value}")
    print(f"- Test size       : {test_size.value}")
    print(f"- Share tolerance : {share_tol.value}")
    print(f"- PCI minimum     : {pci_min.value}")
else:
    display(wg.HBox([n_seed, test_size, share_tol, pci_min],
                    layout=wg.Layout(justify_content="center", flex_flow="row wrap")))

def search_params():
    return {
        "n_seed": n_seed.value,
        "test_size": test_size.value,
        "share_tol": share_tol.value,
        "pci_min": pci_min.value,
    }
SPLIT_UI = ("n_seed", "test_size", "share_tol", "pci_min",
            "style", "box", "search_params")

Parameter Pemisahan Data:
- Jumlah seed     : 200
- Test size       : 0.2
- Share tolerance : 0.01
- PCI minimum     : 0.0


In [5]:
# Memisah data sesuai kriteria
from IPython.utils import io

if globals().get("df") is None:
    df, features = clean_data(load_data(filename))

if EXPORT_MODE:
    # Run the function silently (hides tqdm progress bars and any internal prints)
    with io.capture_output() as captured:
        result = split_data(df, features, filename, search_params())
else:
    # Run normally with the progress bar visible
    result = split_data(df, features, filename, search_params())

split_summary = result["summary"]

# Print the summary outside the block so it always shows up
print(split_summary)

if EXPORT_MODE:
    release("captured", "result", "df", *SPLIT_UI, "SPLIT_UI")
else:
    release("result")

Seed used: 179 (Gap: 0.7556)
Train: 933 rows (79.27%) | Test: 244 rows (20.73%)
Roads: 27 train | 7 test
Saved to: C:\Users\gearb\Documents\pci-ml\datasets\pci-ml-26\n200_t0.2_s0.01_p0.0


#### 3. Pelatihan model *machine learning*

In [6]:
# Memuat data yang sudah dipisah sesuai kriteria
ckpt_dd = wg.Dropdown(
    options=checkpoint_options(dropdown.value),
    description="Split :",
    layout=wg.Layout(width="700px"),
)

train_df = test_df = meta = None
if CKPT_TAG is not None:
    ckpt_dd.value = CKPT_TAG

def load_selected_ckpt(ckpt_name):
    global train_df, test_df, features, meta
    if ckpt_name is not None:
        train_df, test_df, features, meta = load_checkpoint(dropdown.value, ckpt_name)

def split_info():
    n_tr, n_te = len(train_df), len(test_df)
    total = n_tr + n_te
    return (f"**Train:** {train_df.shape} ({n_tr / total:.2%}) | "
            f"**Test:** {test_df.shape} ({n_te / total:.2%})")

if EXPORT_MODE:
    selected_ckpt = ckpt_dd.value
    load_selected_ckpt(selected_ckpt)
    display(Markdown(f"**Split terpilih:** `{selected_ckpt}`\n\n{split_info()}"))
else:
    display(ckpt_dd)
    ckpt_handle = display(Markdown(""), display_id=True)

    def on_ckpt(change):
        load_selected_ckpt(change["new"])
        ckpt_handle.update(Markdown(f"{split_info()}\n\n"))

    ckpt_dd.observe(on_ckpt, names="value")
    if ckpt_dd.value is not None:
        on_ckpt({"new": ckpt_dd.value})

# release leftovers now that train_df/test_df are loaded from disk
release("df",
        *(["ckpt_dd", "load_selected_ckpt", "selected_ckpt", "split_info"]
          if EXPORT_MODE else []))

Loading Checkpoints:   0%|          | 0/1 [00:00<?, ?it/s]

**Split terpilih:** `n200_t0.2_s0.01_p0.0`

**Train:** (933, 59) (79.27%) | **Test:** (244, 59) (20.73%)

### ***Source code* modul dan fungsi**

In [7]:
# import os
# from pathlib import Path
# from IPython.display import display, Markdown

# # Path to your src directory
# src_dir = Path("src")

# # Loop through all Python files and render them as Markdown code blocks
# for filepath in src_dir.rglob("*.py"): # rglob searches subdirectories too
    
#     # Ignore any files inside Jupyter checkpoint folders
#     if ".ipynb_checkpoints" in filepath.parts:
#         continue

#     with open(filepath, "r", encoding="utf-8") as f:
#         code = f.read()
    
#     # Creates a nice header for the filename and formats the code
#     display(Markdown(f"#### `{filepath.name}`\n```python\n{code}\n```\n---"))